# Bias Experiment Visualization Notebook

This notebook turns completed LLM audit and Scholarity/provider runs into thesis-ready analysis tables and visual comparisons for the master's thesis **Design and Implementation of a Tool for Exploring Bias in AI-based Scientific Literature Search**.

It is intentionally read-only with respect to the production application: it reads SQLite records and run artifacts, then writes figures and CSV tables under `notebooks/figures/` and `notebooks/tables/`.

## Experiment Design

The analysis uses paired queries. Each pair has a baseline/control query, a bias-triggering variant, and an intended bias dimension. Result rows are mapped to all relevant pairs, so a query such as `computer vision` can appear in both the core experiment and the venue/prestige experiment.

Interpretation should be cautious. The charts show shifts in retrieved or generated bibliographic records based on available metadata. Missing metadata, canonical mismatch, and enrichment failure are treated as evidence of verifiability risk, not as direct proof that a record is fabricated.

## Query Pair Configuration

Manual configuration is below. Leave `LLM_RUN_IDS` and `SCHOLARITY_RUN_IDS` empty to auto-detect relevant completed or partial runs by matching the configured query sets.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

LLM_RUN_IDS = []
SCHOLARITY_RUN_IDS = []
TOP_K = 10
SAVE_FIGURES = True
SAVE_TABLES = True

FIGURES_DIR = PROJECT_ROOT / "notebooks" / "figures"
FIGURES_SELECTED_DIR = PROJECT_ROOT / "notebooks" / "figures_selected"
TABLES_DIR = PROJECT_ROOT / "notebooks" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_SELECTED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures: {FIGURES_DIR}")
print(f"Selected figures: {FIGURES_SELECTED_DIR}")
print(f"Tables: {TABLES_DIR}")

In [ ]:
from bias_experiment_utils import (
    EXPERIMENT_PAIRS,
    build_all_outputs,
    export_selected_figures,
    export_tables,
    list_available_runs,
    load_runs,
    save_required_figures,
)

experiment_pairs = __import__("pandas").DataFrame(EXPERIMENT_PAIRS)
experiment_pairs

## Run Discovery

Runs are discovered from SQLite and from `data/run_artifacts/`. The artifact path is retained for traceability. Auto-detection scores runs by overlap with the expected thesis queries and prefers runs with records.

In [ ]:
available_runs = list_available_runs(PROJECT_ROOT)
display_cols = ["run_id", "run_type", "status", "created_at", "completed_at", "top_k", "query_count", "entity_count", "record_count", "artifact_path"]
available_runs[display_cols].head(40)

## Data Loading and Normalization

The loader reads the durable record-to-query mapping from SQLite and combines result rows with canonical enrichment fields where available. This produces one normalized record DataFrame with model/provider identifiers, bibliographic metadata, enrichment status, metadata completeness, and a conservative bibliographic risk label.

The existing run artifacts remain the source of truth for run provenance and diagnostics. In this project version, the complete record mapping is stored in SQLite, while artifact folders contain manifests, analysis summaries, raw LLM/provider attempts, and enrichment details.

In [ ]:
outputs = build_all_outputs(PROJECT_ROOT, LLM_RUN_IDS, SCHOLARITY_RUN_IDS, TOP_K)

selected_run_ids = outputs["selected_run_ids"]
records = outputs["records"]
mapped_records = outputs["mapped_records"]
query_diagnostics = outputs["query_diagnostics"]

print("Selected run IDs:")
for run_id in selected_run_ids:
    print(" -", run_id)
print(f"Loaded records: {len(records):,}")
print(f"Experiment-mapped rows: {len(mapped_records):,}")
print(f"Unique loaded queries: {records['query'].nunique() if not records.empty else 0}")
records.head()

## Query Matching Diagnostics

This section shows which configured thesis queries were present in the loaded runs. If auto-detection fails in another environment, inspect this table together with the discovered run list above and manually set `LLM_RUN_IDS` / `SCHOLARITY_RUN_IDS`.

In [ ]:
print(query_diagnostics["status"].value_counts().to_string())
query_diagnostics.sort_values(["status", "query"])

## Descriptive Statistics

These counts provide a sanity check before interpreting bias metrics. Uneven counts can reflect model parse failures, provider failures, partial runs, or provider-specific result limits.

In [ ]:
table_record_counts_by_query = outputs["tables"]["table_record_counts_by_query"]
table_record_counts_by_query.head(60)

In [ ]:
records.groupby(["source_system_type", "source_system"], dropna=False).agg(
    records=("record_id", "count"),
    queries=("query", "nunique"),
    metadata_completeness=("metadata_completeness_score", "mean"),
    high_risk_share=("hallucination_risk", "mean"),
).reset_index().sort_values(["source_system_type", "records"], ascending=[True, False])

## Pairwise Baseline vs Variant Analysis

Jaccard similarity and top-K overlap describe how strongly a bias-triggering query changed the result set relative to its baseline. Lower overlap suggests higher query sensitivity. Year deltas, coverage deltas, and risk deltas should be read as metadata-based indicators, not causal claims.

In [ ]:
pairwise = outputs["pairwise"]
pairwise.sort_values(["experiment_set", "pair_id", "source_system_type", "source_system"]).head(80)

## Bias-Specific Analysis

The following tables separate direct metadata from proxy indicators where possible. Geographic buckets use country metadata first and simple title/venue heuristics only when country data is missing. Language buckets use direct language metadata first. Venue, publisher, citation/popularity, and open-access/preprint analyses use concentration and coverage proxies when direct evidence is unavailable.

In [ ]:
metadata = outputs["metadata"]
hallucination = outputs["hallucination"]
geographic = outputs["geographic"]
language = outputs["language"]
concentration = outputs["concentration"]
open_access = outputs["open_access"]
ranking = outputs["ranking"]
concentration_delta = outputs["concentration_delta"]
open_access_delta = outputs["open_access_delta"]
metadata_risk_delta = outputs["metadata_risk_delta"]
geographic_delta = outputs["geographic_delta"]
language_delta = outputs["language_delta"]
candidate_figures = outputs["candidate_figures"]
safe_claims = outputs["safe_claims"]
thematic = outputs["thematic"]

print("Geographic rows:", len(geographic))
print("Language rows:", len(language))
print("Venue/publisher rows:", len(concentration))
print("Open access/preprint rows:", len(open_access))
print("Thematic keyword rows:", len(thematic))
print("Concentration delta rows:", len(concentration_delta))
print("Language delta rows:", len(language_delta))

In [ ]:
geographic.head(40)

In [ ]:
language.head(60)

In [ ]:
concentration.sort_values(["experiment_set", "pair_id", "source_system", "query_type"]).head(80)

In [ ]:
open_access.head(60)

## LLM vs Scholarly Provider Comparison

Where both LLM and Scholarity/provider records exist for the same query, overlap is computed between normalized DOI/title sets. A higher model-provider Jaccard score suggests the model output resembles that provider's result set more closely for the available queries.

In [ ]:
llm_provider_overlap = outputs["llm_provider_overlap"]
provider_comparison = outputs["provider_comparison"]
print("LLM-provider overlap rows:", len(llm_provider_overlap))
llm_provider_overlap.sort_values("jaccard_similarity", ascending=False).head(40)

In [ ]:
provider_comparison.head(60)

## Figures for Thesis

Figures are saved as PNG files under `notebooks/figures/`; bar charts also save SVG companions where possible. The main thesis figures use short model/provider labels, aggregation, top-N filtering, and delta views to avoid unreadable axes. Use the CSV tables for exact values.

In [ ]:
created_figures = save_required_figures(
    figures_dir=FIGURES_DIR,
    mapped=mapped_records,
    pairwise=pairwise,
    metadata=metadata,
    hallucination=hallucination,
    geographic=geographic,
    language=language,
    concentration=concentration,
    open_access=open_access,
    ranking=ranking,
    llm_provider_overlap=llm_provider_overlap,
    provider_comparison=provider_comparison,
    save=SAVE_FIGURES,
)
print(f"Created {len(created_figures)} figures")
for path in created_figures:
    print(path.relative_to(PROJECT_ROOT))

## Figure Selection and Thesis-Safe Claims

Not every generated chart should be included in the thesis. This section scores figures using simple heuristics for readability, interpretability, data completeness, and expected usefulness. Recommended figures are copied into `notebooks/figures_selected/` together with a CSV explanation table.

The safe-claims table phrases conclusions cautiously and links each claim to a supporting figure and table.

In [ ]:
candidate_figures

In [ ]:
recommended_figures = export_selected_figures(candidate_figures, FIGURES_DIR, FIGURES_SELECTED_DIR)
outputs["tables"]["table_recommended_thesis_figures"] = recommended_figures
print(f"Recommended figures copied: {len(recommended_figures)}")
recommended_figures[["figure_file", "metric", "thesis_usefulness_score", "reason", "possible_thesis_claim"]] if not recommended_figures.empty else recommended_figures

In [ ]:
safe_claims

## Exported Tables

CSV tables are saved under `notebooks/tables/`. These tables are the source for exact thesis values; charts are intended for visual comparison.

In [ ]:
export_tables(outputs["tables"], TABLES_DIR, save=SAVE_TABLES)
for name, table in outputs["tables"].items():
    print(f"{name}: {len(table):,} rows")

## Limitations and Interpretation Notes

- Jaccard similarity and top-K overlap indicate result-set sensitivity to query wording; they do not by themselves identify why a shift occurred.
- Geographic and language charts use direct metadata when present. Buckets labelled `heuristic` are approximate text indicators.
- Citation/popularity analysis uses citation counts where enrichment provides them. When citation counts are missing, venue concentration, publisher concentration, repeated canonical works, and overlap are proxy indicators only.
- Hallucination risk is operationalized as bibliographic verifiability risk: enrichment failure, unmatched canonical records, invalid DOI-like values, and missing core metadata.
- Partial LLM runs are included because completed experiment batches in this project may have some failed model calls while still containing usable records.
- Provider runs and LLM audit runs are not identical retrieval processes. Comparisons should be phrased as resemblance or divergence based on available records, not as ground-truth correctness.